# XAI stability

In [ ]:
import sys

!{sys.executable} -m pip install xgboost
!{sys.executable} -m pip install catboost
!{sys.executable} -m pip install nbimporter
!{sys.executable} -m pip install tensorflow
!{sys.executable} -m pip install torch
!{sys.executable} -m pip install captum
!{sys.executable} -m pip install shap

#!git clone https://github.com/AI4LIFE-GROUP/OpenXAI.git
!{sys.executable} -m pip install -e OpenXAI

In [ ]:
import time
import numpy as np
import pandas as pd
import nbimporter

import sklearn.ensemble
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from scipy.sparse.linalg import eigs

import xgboost as xgb
#from catboost import CatBoostClassifier

import lime
import lime.lime_tabular

import shap
shap.initjs()

from captum.attr import IntegratedGradients
from captum.attr import InputXGradient
from captum.attr import DeepLift
from captum.attr import LRP

In [ ]:
import openxai

# Utils
import torch
import pickle

# Data loaders
from openxai.dataloader import return_loaders

# Perturbation methods required for the computation of the relative stability metrics
from openxai.explainers.catalog.perturbation_methods import NormalPerturbation
from openxai.explainers.catalog.perturbation_methods import NewDiscrete_NormalPerturbation

In [ ]:
# Perturbation class parameters
perturbation_mean= 0.0
perturbation_std= 0.001
perturbation_flip_percentage= 0.0001
    
perturbation= NormalPerturbation('tabular',
                                 mean=perturbation_mean,
                                 std_dev=perturbation_std,
                                 flip_percentage=perturbation_flip_percentage)

def generate_mask(explanation, top_k):
    mask_indices= torch.topk(explanation, top_k).indices
    mask= torch.zeros(explanation.shape) > 10
    for i in mask_indices:
        mask[i]= True
    return mask

# Quantitative Metrics -- Auxiliar Methods

In [ ]:
# clip values near to zero in v replacing by eps

# v is a single value (float) or a numpy.ndarray with (n,) shape
# eps is a small number of tolerance limiting what is a small value

# RETURN v clipped

def clip_small_values(v, eps=1e-6):
    
    v_aux= v.copy()
    
    if (type(v_aux)== np.ndarray):
        elements= v_aux.shape[0]

        for i in range(elements):

            if (v_aux[i]< 0 and np.abs(v_aux[i])< eps):
                v_aux[i]= -eps
            elif (v_aux[i]> 0 and v_aux[i]< eps):
                v_aux[i]= eps
    else:
        if (v_aux< 0 and np.abs(v_aux)< eps):
            v_aux[i]= -eps
        elif (v_aux> 0 and v_aux< eps):
            v_aux= eps
                        
    return v_aux

In [ ]:
# returns the square of the difference of any two quantities v1 and v2.

def square_difference(v1, v2):
    
    # arrays can be flattened, so long as ordering is preserved
    v1_flat= np.asarray(v1).flatten()
    v2_flat= np.asarray(v2).flatten()
    
    dif_flat= (v1_flat - v2_flat)
    
    return np.power(dif_flat, 2)

In [ ]:
# returns the Lp norm of the difference between v1 and v2.
# normalizes the difference between v1 and v2 by v1 (adapted; Agarwal, Chirag, et al., 2022)

def lp_norm_dif(v1, v2, p_norm=2, eps=1e-6, norm:bool=True):
    
    # arrays can be flattened, so long as ordering is preserved
    v1_flat= np.asarray(v1).flatten()
    v2_flat= np.asarray(v2).flatten()
    
    dif_flat= (v1_flat - v2_flat)
    
    #if (norm==True): print('dif before div', dif_flat)
    
    if (norm==True):
        #v1_flat= np.clip(v1_flat, eps, None)
        v1_flat= clip_small_values(v1_flat, eps)
        
        dif_flat= np.divide(dif_flat, v1_flat, out=np.zeros_like(v1_flat), where=v1_flat!=0)
        
        #v2_flat= clip_small_values(v2_flat, eps)
        #dif_flat= 1 - np.divide(v2_flat, v1_flat, out=np.zeros_like(v1_flat), where=v1_flat!=0)
        
    #if (norm==True): print('dif afterr div', dif_flat)

    return np.linalg.norm(dif_flat, ord=p_norm)

In [ ]:
# RETURN tensor_x and tensor_y datsets (tensors) according to euclidean distance ordering from target_x

def distance_ordering(tensor_x, tensor_y, target_x):
    
    # Calculate Euclidean distances for each row
    distances= torch.norm((tensor_x - target_x), dim=1)

    # Sort the data tensor based on distances
    sorted_indices= torch.argsort(distances)
    
    sorted_x= tensor_x[sorted_indices]
    sorted_y= tensor_y[sorted_indices]
    
    return sorted_x, sorted_y

In [ ]:
# Remove a set of rows in a tensor dataset by index

# dataset is a n elements dataset
# index_to_remove is a m elements tensor with the indexes to remove

# RETURN a subset form dataset without the index_to_remove instances
    
def remove_tensor_row_by_indexset(dataset, index_to_remove):

    n_rows= index_to_remove.shape[0]
    
    index_to_remove= index_to_remove.sort().values
    
    subset= dataset.clone()
    
    for i in range(n_rows):
        row_exclude= index_to_remove[i]-i
    
        subset= torch.cat((subset[:row_exclude],subset[row_exclude+1:]))

    return subset

In [ ]:
# Get a subset from a dataset with at least n_elements

# x is a tensor instance
# x_class is a tensor with the class of x
# dataset is a m elements tensor dataset
# dataset_class is a m elements tensor with the predicted 
# n_elements is an integer indicating the size of the subset

# RETURN two tensor subsets (from dataset and dataset_class) with n_elements ordered first by class 
#        (same from x) and then by distance from x

def get_subsets(x, x_class, dataset, dataset_class, n_elements):
    
    data_size= dataset.shape[0]
    
    if (data_size< n_elements):
        raise ValueError("Data size must be greater than n_elements!")
    else:
        # order the dataset and dataset_class by distance from x
        dataset_order, dataset_class_order= distance_ordering(dataset, dataset_class, x.unsqueeze(0))
        
        # get the subset with first num_perts points by the same x class
        ind_same_class= (x_class == dataset_class_order).nonzero()[:n_elements].squeeze()
        
        subset= torch.index_select(input=dataset_order, dim=0, index=ind_same_class)
        subset_class= torch.index_select(input=dataset_class_order, dim=0, index=ind_same_class)
        
        # if there are no elements enough in dataset matching with x_class, we complete the n_elements
        # of subset with the first instances of the ordered dataset
        if (subset.shape[0]< n_elements):
            last= n_elements - subset.shape[0]
            
            if (ind_same_class.numel()== 1): # avoid a breaking when only one element matches with x_class
                ind_same_class= torch.tensor([ind_same_class])
            
            dataset_order= remove_tensor_row_by_indexset(dataset_order, ind_same_class)
            dataset_class_order= remove_tensor_row_by_indexset(dataset_class_order, ind_same_class)
            
            dataset_order= dataset_order[0:last,:]
            dataset_class_order= dataset_class_order[0:last]
            
            subset= torch.cat((subset, dataset_order))
            subset_class= torch.cat((subset_class, dataset_class_order))
            
            
        return subset, subset_class

In [ ]:
# compute norm between predictions per perturbation - RIS

def ris_measure(x_data, x_pert, exp_data, exp_pert, p_norm=2, eps=1e-6):
    
    x_dif_norm= lp_norm_dif(x_data, x_pert, p_norm=p_norm, eps=eps, norm=True)
    #x_dif_norm= np.clip(x_dif_norm, eps, None)
    x_dif_norm= clip_small_values(x_dif_norm, eps)
    
    exp_dif_norm= lp_norm_dif(exp_data, exp_pert, p_norm=p_norm, eps=eps, norm=True)
    
    stability_measure= np.divide(exp_dif_norm, x_dif_norm, where=x_dif_norm!=0)
    
    """
    print('x_data', x_data)
    print('x_pert', x_pert)
    print('x_dif_norm', x_dif_norm)
    print('exp_ dif_norm', exp_dif_norm)
    print('stab_measure', stability_measure)
    """
    
    return stability_measure

In [ ]:
# compute norm between representations - ROS

# x_data and x_pert must to be pd.DataFrame row individual instances with column names

def ros_measure(model, x_data, x_pert, exp_data, exp_pert, p_norm=2, eps=1e-6):
        
    fx_data= model.predict_proba(x_data)
    fx_pert= model.predict_proba(x_pert)
    
    fx_dif_norm= lp_norm_dif(fx_data, fx_pert, p_norm=p_norm, eps=eps, norm=True)
    #fx_dif_norm= np.clip(fx_dif_norm, eps, None)
    fx_dif_norm= clip_small_values(fx_dif_norm, eps)
    
    exp_dif_norm= lp_norm_dif(exp_data, exp_pert, p_norm=p_norm, eps=eps, norm=True)

    stability_measure= np.divide(exp_dif_norm, fx_dif_norm, where=fx_dif_norm!=0)
    
    """
    print('x_data', x_data)
    print('x_pert', x_pert)
    print('fx_data', fx_data)
    print('fx_pert', fx_pert)
    print('x_dif_norm', fx_dif_norm)
    print('exp_ dif_norm', exp_dif_norm)
    print('stab_measure', stability_measure)
    """
    
    return stability_measure

In [ ]:
# bring explanations into data order (since LIME automatically orders according to highest importance)

def lime_exp_in_data_order(lime_exp, num_fts):
    
    exp= np.zeros(num_fts)

    for k, v in lime_exp.local_exp[1]:
        exp[k]= v

    return exp

# Metric -- Relative Input/Output Stability -- RIS / ROS

In [ ]:
# Relative Input/Output Stability
# model is a treined classifier
# data is a preprocessed Pandas DataFrame -- model's training data
# labels are the data labels (Pandas DataFrame)
# perturbation is a OpenXAI perturbation object
# descriptor define the parameters to explanations and data perturbations
# cat_fts list indicating the categorical columns. if empty metric will consider all features as numeric
# model_is_NN True if model is a sklearn MLPClassifier, False if it is not 

# approximates the maximum L-p distance between explanations in a neighborhood around input x
# RETURN RIS max/mean and ROS max/mean metrics for T-Exp, SHAP, and LIME.

def relative_stability(model, data, labels, perturbation, descriptor, cat_fts=[], is_model_NN:bool=False):
    
    tensor_train= torch.from_numpy(data.values)
    tensor_labels= torch.from_numpy(labels.values.ravel().astype(int))
    
    # ------------ data reduction for testing
    tensor_train = tensor_train[0:100, :]      #[91:92, :] # border instance to xgb_model #
    tensor_labels= tensor_labels[0:100]        #[91:92]
    # ---------------------------------------
    
    t_ris_max_ratios= []
    shap_ris_max_ratios= []
    lime_ris_max_ratios= []
    
    t_ris_mean_ratios= []
    shap_ris_mean_ratios= []
    lime_ris_mean_ratios= []
    
    t_ros_max_ratios= []
    shap_ros_max_ratios= []
    lime_ros_max_ratios= []
    
    t_ros_mean_ratios= []
    shap_ros_mean_ratios= []
    lime_ros_mean_ratios= []
    
    if (is_model_NN==True):
        # convert a scikit-learn NN model to a PyTorch NN model used in captum
        nn_pytorch_model= sklearn_to_pytorch_NN(model, data.shape[1])
            
        itGd_ris_max_ratios= []
        iXGd_ris_max_ratios= []
        dLif_ris_max_ratios= []
        lwrp_ris_max_ratios= []

        itGd_ris_mean_ratios= []
        iXGd_ris_mean_ratios= []
        dLif_ris_mean_ratios= []
        lwrp_ris_mean_ratios= []

        itGd_ros_max_ratios= []
        iXGd_ros_max_ratios= []
        dLif_ros_max_ratios= []
        lwrp_ros_max_ratios= []

        itGd_ros_mean_ratios= []
        iXGd_ros_mean_ratios= []
        dLif_ros_mean_ratios= []
        lwrp_ros_mean_ratios= []
        
    
    ohe_model= clone(model)
    
    if (np.asarray(cat_fts).shape[0]> 0):
        num_ohe_data= ohe_cat_to_numerical_simulator(data, cat_fts, delta=descriptor['ohe_delta'], 
                                                     rand_seed=True)
        ohe_model.fit(num_ohe_data, labels.values.ravel())
        retrained= True
    else:
        ohe_model.fit(data, labels.values.ravel())
        retrained= False
        
    
    for i_data, x_data in enumerate(tensor_train):
        # here x_data is tensor_train[i_data]
        
        # x_data and its label as pd.DataFrame
        target_i= pd.DataFrame(data=[x_data.numpy()], columns=data.columns)
        target_l= pd.DataFrame(data=[np.int64(tensor_labels[i_data])], columns=labels.columns)
        
        # data point prediction
        y_pred= torch.from_numpy(model.predict(target_i).astype(int))
        
        # ignore temporarily warnings related to feature names
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message="X does not have valid feature names")
            
            # ------------------------------------ x_data explanation
            # data point explanation -- T-Exp
            t_x_exp, t_x_ft, t_x_sh= categorical_taylor_explainer(ohe_model, data, labels, 
                                                                  target_i, target_l, cat_cols=cat_fts, 
                                              h_min=descriptor['h_min'], h_max=descriptor['h_max'],
                                              eps=descriptor['jacobian_eps'], max_itr=descriptor['max_itr'],
                                              finite_diff_version=descriptor['finite_diff_version'],
                                              delta=descriptor['ohe_delta'], e_x=descriptor['e_x'], 
                                              angle=descriptor['angle'], retrained_ohe_model=retrained, 
                                              verbose=False)
            
            t_x_exp= torch.from_numpy(t_x_exp)


            # data point explanation -- SHAP
            if isinstance(model, xgb.XGBModel):
                shap_exp_gen= shap.TreeExplainer(model)
            else:
                shap_exp_gen= shap.Explainer(model.predict, data)
            
            shap_x_exp= shap_exp_gen(target_i)
            shap_x_exp= (torch.from_numpy(shap_x_exp.values)).squeeze()


            # data point explanation -- LIME
            lime_exp_gen= lime.lime_tabular.LimeTabularExplainer(training_data=np.asarray(data), 
                                                                 feature_names=np.asarray(data.columns), 
                                                                 training_labels=labels.values.ravel().astype(int), 
                                                                 class_names=np.asarray([0,1]), 
                                                                 mode='classification', 
                                                                 discretize_continuous=False, verbose=False)

            lime_scores= lime_exp_gen.explain_instance(data_row=np.asarray(target_i)[0], 
                                                       predict_fn=model.predict_proba, 
                                                       num_features=data.shape[1])

            lime_x_exp= torch.from_numpy(lime_exp_in_data_order(lime_scores, data.shape[1]))
        
        # reset the warning settings
        warnings.resetwarnings()
        
        
        # data point explanation -- Gradient-based methods
        if (is_model_NN==True):
            x_data_tensor= torch.tensor(np.asarray(target_i), dtype=torch.float32)
            x_data_tensor.requires_grad= True
            
            # ignore non-important warnings temporarily
            with warnings.catch_warnings():
                warnings.filterwarnings("ignore", message="Setting forward, backward hooks and attributes")
            
                itGd= IntegratedGradients(nn_pytorch_model)
                itGd_x_exp= (itGd.attribute(x_data_tensor)).squeeze().detach()

                iXGd= InputXGradient(nn_pytorch_model)
                iXGd_x_exp= (iXGd.attribute(x_data_tensor.unsqueeze(0))).squeeze().detach()

                dLif= DeepLift(nn_pytorch_model)
                dLif_x_exp= (dLif.attribute(x_data_tensor.unsqueeze(0))).squeeze().detach()

                lwrp= LRP(nn_pytorch_model)
                lwrp_x_exp= (lwrp.attribute(x_data_tensor.unsqueeze(0))).squeeze().detach()
                
            # reset the warning settings
            warnings.resetwarnings()
            

        # ------------------------------------ x_data perturbation
        # data point perturbation
        x_pert_samples= perturbation.get_perturbed_inputs(original_sample=x_data,
                                                          feature_mask=descriptor['mask'],
                                                          num_samples=descriptor['num_samples'],
                                                          max_distance=descriptor['pert_max_distance'],
                                                          feature_metadata=descriptor['feature_metadata'])

        # --- take the closest num_perts points to x_data that have the same predicted class label to x_data
        y_pert_preds= torch.from_numpy(model.predict(pd.DataFrame(data=x_pert_samples.numpy(),
                                                                 columns=data.columns)).astype(int))
        
        # get only the first num_perts points ordered by class and distance from x_data
        x_pert_samples, y_pert_preds= get_subsets(x_data, y_pred, x_pert_samples, y_pert_preds, 
                                                  descriptor['num_perts'])
        
        
        # ------------------------------------ explain each x_data perturbation
        t_exp_pert_samples= torch.zeros_like(x_pert_samples)
        shap_exp_pert_samples= torch.zeros_like(x_pert_samples)
        lime_exp_pert_samples= torch.zeros_like(x_pert_samples)
        
        if (is_model_NN==True):
            itGd_exp_pert_samples= torch.zeros_like(x_pert_samples)
            iXGd_exp_pert_samples= torch.zeros_like(x_pert_samples)
            dLif_exp_pert_samples= torch.zeros_like(x_pert_samples)
            lwrp_exp_pert_samples= torch.zeros_like(x_pert_samples)
        
        
        t_x_ris_ratios= []
        shap_x_ris_ratios= []
        lime_x_ris_ratios= []
        
        t_x_ros_ratios= []
        shap_x_ros_ratios= []
        lime_x_ros_ratios= []
        
        if (is_model_NN==True):
            itGd_x_ris_ratios= []
            iXGd_x_ris_ratios= []
            dLif_x_ris_ratios= []
            lwrp_x_ris_ratios= []

            itGd_x_ros_ratios= []
            iXGd_x_ros_ratios= []
            dLif_x_ros_ratios= []
            lwrp_x_ros_ratios= []
        
        """
        print('\nInstance', i_data)
        print('x', x_data)
        print('y_pred', y_pred)
        print('tx_exp', t_x_exp)
        print('shap_exp', shap_x_exp)
        print('lime_exp', lime_x_exp)
        if (is_model_NN==True):
            print('itGd_x_exp', itGd_x_exp)
            print('iXGd_x_exp', iXGd_x_exp)
            print('dLif_x_exp', dLif_x_exp)
            print('lwrp_x_exp', lwrp_x_exp)
        print('x_pert_samples\n', x_pert_samples)
        print('y_pert_preds', y_pert_preds)
        #"""
        
        # For each perturbation, calculate the explanation
        for i, x_pert in enumerate(x_pert_samples):
            
            df_x_pert= pd.DataFrame(data=[x_pert.numpy()], columns=data.columns)
            df_y_pert= pd.DataFrame(data=[np.int64(y_pert_preds[i])], columns=labels.columns)
            
            # ignore temporarily warnings related to feature names
            with warnings.catch_warnings():
                warnings.filterwarnings("ignore", message="X does not have valid feature names")
            
                # ------------------------------------ x_pert explanation
                # perturbed data point explanation -- T-Exp
                t_exp, t_ft, t_sh= categorical_taylor_explainer(ohe_model, data, labels, 
                                                                df_x_pert, df_y_pert, cat_cols=cat_fts, 
                                                h_min=descriptor['h_min'], h_max=descriptor['h_max'],
                                                eps=descriptor['jacobian_eps'], max_itr=descriptor['max_itr'],
                                                finite_diff_version=descriptor['finite_diff_version'],
                                                delta=descriptor['ohe_delta'], e_x=descriptor['e_x'], 
                                                angle=descriptor['angle'], retrained_ohe_model=retrained, 
                                                verbose=False)
            
                t_exp_pert_samples[i, :]= torch.from_numpy(t_exp)
            
            
                # perturbed data point explanation -- SHAP
                if isinstance(model, xgb.XGBModel):
                    shap_exp_gen= shap.TreeExplainer(model)
                else:
                    shap_exp_gen= shap.Explainer(model.predict, data)
                    
                shap_exp= shap_exp_gen(df_x_pert)
                shap_exp_pert_samples[i, :]= torch.from_numpy(shap_exp.values)


                # perturbed data point explanation -- LIME
                #lime_exp_gen= lime.lime_tabular.LimeTabularExplainer(training_data=np.asarray(data), 
                #                                                     feature_names=np.asarray(data.columns), 
                #                                                     training_labels=labels.values.ravel().astype(int), 
                #                                                     class_names=np.asarray([0,1]), 
                #                                                     mode='classification', 
                #                                                     discretize_continuous=False, verbose=False)

                lime_scores= lime_exp_gen.explain_instance(data_row=np.asarray(df_x_pert)[0], 
                                                           predict_fn=model.predict_proba, 
                                                           num_features=data.shape[1])

                lime_exp_pert_samples[i, :]= torch.from_numpy(lime_exp_in_data_order(lime_scores, data.shape[1]))
            
            # reset the warning settings
            warnings.resetwarnings()
            
            
            # perturbed data point explanation -- Gradient-based methods
            if (is_model_NN==True):
                x_pert_tensor= torch.tensor(np.asarray(df_x_pert), dtype=torch.float32)
                x_pert_tensor.requires_grad= True
                
                # ignore non-important warnings temporarily
                with warnings.catch_warnings():
                    warnings.filterwarnings("ignore", message="Setting forward, backward hooks and attributes")
                
                    itGd_exp_pert_samples[i, :]= (itGd.attribute(x_pert_tensor)).squeeze().detach()
                    iXGd_exp_pert_samples[i, :]= (iXGd.attribute(x_pert_tensor.unsqueeze(0))).squeeze().detach()
                    dLif_exp_pert_samples[i, :]= (dLif.attribute(x_pert_tensor.unsqueeze(0))).squeeze().detach()
                    lwrp_exp_pert_samples[i, :]= (lwrp.attribute(x_pert_tensor.unsqueeze(0))).squeeze().detach()
                
                # reset the warning settings
                warnings.resetwarnings()
        
    
            # ------------------------------------ get stability for each explanator and x_data perturbation
            t_ris_measure= ris_measure(x_data, x_pert, 
                                       t_x_exp, t_exp_pert_samples[i], 
                                       p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
            
            shap_ris_measure= ris_measure(x_data, x_pert, 
                                          shap_x_exp, shap_exp_pert_samples[i], 
                                          p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
            
            lime_ris_measure= ris_measure(x_data, x_pert, 
                                          lime_x_exp, lime_exp_pert_samples[i], 
                                          p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
            
            if (is_model_NN==True):
                itGd_ris_measure= ris_measure(x_data, x_pert, 
                                             itGd_x_exp, itGd_exp_pert_samples[i], 
                                             p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
                
                iXGd_ris_measure= ris_measure(x_data, x_pert, 
                                              iXGd_x_exp, iXGd_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
                
                dLif_ris_measure= ris_measure(x_data, x_pert, 
                                              dLif_x_exp, dLif_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
                
                lwrp_ris_measure= ris_measure(x_data, x_pert, 
                                              lwrp_x_exp, lwrp_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
                
            
            df_x_data= target_i.copy()

            t_ros_measure= ros_measure(model, df_x_data, df_x_pert, 
                                       t_x_exp, t_exp_pert_samples[i], 
                                       p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
            
            shap_ros_measure= ros_measure(model, df_x_data, df_x_pert, 
                                          shap_x_exp, shap_exp_pert_samples[i], 
                                          p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
            
            lime_ros_measure= ros_measure(model, df_x_data, df_x_pert, 
                                          lime_x_exp, lime_exp_pert_samples[i], 
                                          p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
            
            if (is_model_NN==True):
                itGd_ros_measure= ros_measure(model, df_x_data, df_x_pert, 
                                              itGd_x_exp, itGd_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
                
                iXGd_ros_measure= ros_measure(model, df_x_data, df_x_pert, 
                                              iXGd_x_exp, iXGd_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
                
                dLif_ros_measure= ros_measure(model, df_x_data, df_x_pert, 
                                              dLif_x_exp, dLif_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
                
                lwrp_ros_measure= ros_measure(model, df_x_data, df_x_pert, 
                                              lwrp_x_exp, lwrp_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
        
            
            # --- stability measures for each x_data perturbation --- one processing cicle
            # RIS
            t_x_ris_ratios.append(t_ris_measure)
            shap_x_ris_ratios.append(shap_ris_measure)
            lime_x_ris_ratios.append(lime_ris_measure)
            
            # ROS
            t_x_ros_ratios.append(t_ros_measure)
            shap_x_ros_ratios.append(shap_ros_measure)
            lime_x_ros_ratios.append(lime_ros_measure)
            
            if (is_model_NN==True):
                itGd_x_ris_ratios.append(itGd_ris_measure)
                iXGd_x_ris_ratios.append(iXGd_ris_measure)
                dLif_x_ris_ratios.append(dLif_ris_measure)
                lwrp_x_ris_ratios.append(lwrp_ris_measure)

                itGd_x_ros_ratios.append(itGd_ros_measure)
                iXGd_x_ros_ratios.append(iXGd_ros_measure)
                dLif_x_ros_ratios.append(dLif_ros_measure)
                lwrp_x_ros_ratios.append(lwrp_ros_measure)
                
        """
        print('t_exp_pert_samples\n', t_exp_pert_samples)
        print('shap_exp_pert_samples\n', shap_exp_pert_samples)
        print('lime_exp_pert_samples\n', lime_exp_pert_samples)
        if (is_model_NN==True):
            print('itGd_exp_pert_samples\n', itGd_exp_pert_samples)
            print('iXGd_exp_pert_samples\n', iXGd_exp_pert_samples)
            print('dLif_exp_pert_samples\n', dLif_exp_pert_samples)
            print('lwrp_exp_pert_samples\n', lwrp_exp_pert_samples)
        #"""
        """  
        print('ris t_exp\n', t_x_ris_ratios)
        print('ris shap\n', shap_x_ris_ratios)
        print('ris lime\n', lime_x_ris_ratios)
        #"""
        
        # --- append only the max/mean value related to each x_data processed
        # max values
        t_ris_max_ratios.append(t_x_ris_ratios[np.argmax(t_x_ris_ratios)])
        shap_ris_max_ratios.append(shap_x_ris_ratios[np.argmax(shap_x_ris_ratios)])
        lime_ris_max_ratios.append(lime_x_ris_ratios[np.argmax(lime_x_ris_ratios)])

        t_ros_max_ratios.append(t_x_ros_ratios[np.argmax(t_x_ros_ratios)])
        shap_ros_max_ratios.append(shap_x_ros_ratios[np.argmax(shap_x_ros_ratios)])
        lime_ros_max_ratios.append(lime_x_ros_ratios[np.argmax(lime_x_ros_ratios)])
        
        # mean values
        t_ris_mean_ratios.append(np.mean(t_x_ris_ratios))
        shap_ris_mean_ratios.append(np.mean(shap_x_ris_ratios))
        lime_ris_mean_ratios.append(np.mean(lime_x_ris_ratios))

        t_ros_mean_ratios.append(np.mean(t_x_ros_ratios))
        shap_ros_mean_ratios.append(np.mean(shap_x_ros_ratios))
        lime_ros_mean_ratios.append(np.mean(lime_x_ros_ratios))
        
        if (is_model_NN==True):
            # max values
            itGd_ris_max_ratios.append(itGd_x_ris_ratios[np.argmax(itGd_x_ris_ratios)])
            iXGd_ris_max_ratios.append(iXGd_x_ris_ratios[np.argmax(iXGd_x_ris_ratios)])
            dLif_ris_max_ratios.append(dLif_x_ris_ratios[np.argmax(dLif_x_ris_ratios)])
            lwrp_ris_max_ratios.append(lwrp_x_ris_ratios[np.argmax(lwrp_x_ris_ratios)])

            itGd_ros_max_ratios.append(itGd_x_ros_ratios[np.argmax(itGd_x_ros_ratios)])
            iXGd_ros_max_ratios.append(iXGd_x_ros_ratios[np.argmax(iXGd_x_ros_ratios)])
            dLif_ros_max_ratios.append(dLif_x_ros_ratios[np.argmax(dLif_x_ros_ratios)])
            lwrp_ros_max_ratios.append(lwrp_x_ros_ratios[np.argmax(lwrp_x_ros_ratios)])

            # mean values
            itGd_ris_mean_ratios.append(np.mean(itGd_x_ris_ratios))
            iXGd_ris_mean_ratios.append(np.mean(iXGd_x_ris_ratios))
            dLif_ris_mean_ratios.append(np.mean(dLif_x_ris_ratios))
            lwrp_ris_mean_ratios.append(np.mean(lwrp_x_ris_ratios))

            itGd_ros_mean_ratios.append(np.mean(itGd_x_ros_ratios))
            iXGd_ros_mean_ratios.append(np.mean(iXGd_x_ros_ratios))
            dLif_ros_mean_ratios.append(np.mean(dLif_x_ros_ratios))
            lwrp_ros_mean_ratios.append(np.mean(lwrp_x_ros_ratios))
           
    # ------------------------------------ RETURN ratios considering all data processed
    t_ris_max = t_ris_max_ratios[np.argmax(t_ris_max_ratios)]
    shap_ris_max= shap_ris_max_ratios[np.argmax(shap_ris_max_ratios)]
    lime_ris_max= lime_ris_max_ratios[np.argmax(lime_ris_max_ratios)]
    
    t_ris_max_std = np.std(t_ris_max_ratios)
    shap_ris_max_std= np.std(shap_ris_max_ratios)
    lime_ris_max_std= np.std(lime_ris_max_ratios)
    
    t_ros_max = t_ros_max_ratios[np.argmax(t_ros_max_ratios)]
    shap_ros_max= shap_ros_max_ratios[np.argmax(shap_ros_max_ratios)]
    lime_ros_max= lime_ros_max_ratios[np.argmax(lime_ros_max_ratios)]
    
    t_ros_max_std = np.std(t_ros_max_ratios)
    shap_ros_max_std= np.std(shap_ros_max_ratios)
    lime_ros_max_std= np.std(lime_ros_max_ratios)
    
        
    t_ris_mean = np.mean(t_ris_mean_ratios)
    shap_ris_mean= np.mean(shap_ris_mean_ratios)
    lime_ris_mean= np.mean(lime_ris_mean_ratios)
    
    t_ris_mean_std = np.std(t_ris_mean_ratios)
    shap_ris_mean_std= np.std(shap_ris_mean_ratios)
    lime_ris_mean_std= np.std(lime_ris_mean_ratios)
    
    t_ros_mean = np.mean(t_ros_mean_ratios)
    shap_ros_mean= np.mean(shap_ros_mean_ratios)
    lime_ros_mean= np.mean(lime_ros_mean_ratios)
    
    t_ros_mean_std = np.std(t_ros_mean_ratios)
    shap_ros_mean_std= np.std(shap_ros_mean_ratios)
    lime_ros_mean_std= np.std(lime_ros_mean_ratios)
    
    if (is_model_NN==True):
        itGd_ris_max= itGd_ris_max_ratios[np.argmax(itGd_ris_max_ratios)]
        iXGd_ris_max= iXGd_ris_max_ratios[np.argmax(iXGd_ris_max_ratios)]
        dLif_ris_max= dLif_ris_max_ratios[np.argmax(dLif_ris_max_ratios)]
        lwrp_ris_max= lwrp_ris_max_ratios[np.argmax(lwrp_ris_max_ratios)]

        itGd_ris_max_std= np.std(itGd_ris_max_ratios)
        iXGd_ris_max_std= np.std(iXGd_ris_max_ratios)
        dLif_ris_max_std= np.std(dLif_ris_max_ratios)
        lwrp_ris_max_std= np.std(lwrp_ris_max_ratios)

        itGd_ros_max= itGd_ros_max_ratios[np.argmax(itGd_ros_max_ratios)]
        iXGd_ros_max= iXGd_ros_max_ratios[np.argmax(iXGd_ros_max_ratios)]
        dLif_ros_max= dLif_ros_max_ratios[np.argmax(dLif_ros_max_ratios)]
        lwrp_ros_max= lwrp_ros_max_ratios[np.argmax(lwrp_ros_max_ratios)]

        itGd_ros_max_std= np.std(itGd_ros_max_ratios)
        iXGd_ros_max_std= np.std(iXGd_ros_max_ratios)
        dLif_ros_max_std= np.std(dLif_ros_max_ratios)
        lwrp_ros_max_std= np.std(lwrp_ros_max_ratios)
        
        
        itGd_ris_mean= np.mean(itGd_ris_mean_ratios)
        iXGd_ris_mean= np.mean(iXGd_ris_mean_ratios)
        dLif_ris_mean= np.mean(dLif_ris_mean_ratios)
        lwrp_ris_mean= np.mean(lwrp_ris_mean_ratios)

        itGd_ris_mean_std= np.std(itGd_ris_mean_ratios)
        iXGd_ris_mean_std= np.std(iXGd_ris_mean_ratios)
        dLif_ris_mean_std= np.std(dLif_ris_mean_ratios)
        lwrp_ris_mean_std= np.std(lwrp_ris_mean_ratios)

        itGd_ros_mean= np.mean(itGd_ros_mean_ratios)
        iXGd_ros_mean= np.mean(iXGd_ros_mean_ratios)
        dLif_ros_mean= np.mean(dLif_ros_mean_ratios)
        lwrp_ros_mean= np.mean(lwrp_ros_mean_ratios)

        itGd_ros_mean_std= np.std(itGd_ros_mean_ratios)
        iXGd_ros_mean_std= np.std(iXGd_ros_mean_ratios)
        dLif_ros_mean_std= np.std(dLif_ros_mean_ratios)
        lwrp_ros_mean_std= np.std(lwrp_ros_mean_ratios)
        
        
    results= {
        't_exp_ris_max': t_ris_max, 'std(t_exp_ris_max)': t_ris_max_std,
        'shap_ris_max': shap_ris_max, 'std(shap_ris_max)': shap_ris_max_std,
        'lime_ris_max': lime_ris_max, 'std(lime_ris_max)': lime_ris_max_std,
        't_exp_ris_mean': t_ris_mean, 'std(t_exp_ris_mean)': t_ris_mean_std,
        'shap_ris_mean': shap_ris_mean, 'std(shap_ris_mean)': shap_ris_mean_std,
        'lime_ris_mean': lime_ris_mean, 'std(lime_ris_mean)': lime_ris_mean_std,
        't_exp_ros_max': t_ros_max, 'std(t_exp_ros_max)': t_ros_max_std,
        'shap_ros_max': shap_ros_max, 'std(shap_ros_max)': shap_ros_max_std,
        'lime_ros_max': lime_ros_max, 'std(lime_ros_max)': lime_ros_max_std,
        't_exp_ros_mean': t_ros_mean, 'std(t_exp_ros_mean)': t_ros_mean_std,
        'shap_ros_mean': shap_ros_mean, 'std(shap_ros_mean)': shap_ros_mean_std,
        'lime_ros_mean': lime_ros_mean, 'std(lime_ros_mean)': lime_ros_mean_std
    }

    if (is_model_NN==True):
        results_grad= {
            'itGd_ris_max': itGd_ris_max, 'std(itGd_ris_max)': itGd_ris_max_std,
            'iXGd_ris_max': iXGd_ris_max, 'std(iXGd_ris_max)': iXGd_ris_max_std,
            'dLif_ris_max': dLif_ris_max, 'std(dLif_ris_max)': dLif_ris_max_std,
            'lwrp_ris_max': lwrp_ris_max, 'std(lwrp_ris_max)': lwrp_ris_max_std,
            'itGd_ris_mean': itGd_ris_mean, 'std(itGd_ris_mean)': itGd_ris_mean_std,
            'iXGd_ris_mean': iXGd_ris_mean, 'std(iXGd_ris_mean)': iXGd_ris_mean_std,
            'dLif_ris_mean': dLif_ris_mean, 'std(dLif_ris_mean)': dLif_ris_mean_std,
            'lwrp_ris_mean': lwrp_ris_mean, 'std(lwrp_ris_mean)': lwrp_ris_mean_std,
            'itGd_ros_max': itGd_ros_max, 'std(itGd_ros_max)': itGd_ros_max_std,
            'iXGd_ros_max': iXGd_ros_max, 'std(iXGd_ros_max)': iXGd_ros_max_std,
            'dLif_ros_max': dLif_ros_max, 'std(dLif_ros_max)': dLif_ros_max_std,
            'lwrp_ros_max': lwrp_ros_max, 'std(lwrp_ros_max)': lwrp_ros_max_std,
            'itGd_ros_mean': itGd_ros_mean, 'std(itGd_ros_mean)': itGd_ros_mean_std,
            'iXGd_ros_mean': iXGd_ros_mean, 'std(iXGd_ros_mean)': iXGd_ros_mean_std,
            'dLif_ros_mean': dLif_ros_mean, 'std(dLif_ros_mean)': dLif_ros_mean_std,
            'lwrp_ros_mean': lwrp_ros_mean, 'std(lwrp_ros_mean)': lwrp_ros_mean_std 
        }

        results.update(results_grad)

    # the max/mean stability ratios
    return results

# Metric -- Run Explanation Stability -- RES

In [ ]:
# Run Explanation Stability
# model is a treined classifier
# data is a preprocessed Pandas DataFrame -- model's training data
# labels are the data labels (Pandas DataFrame)
# descriptor define the parameters to explanations and data perturbations
# cat_fts list indicating the categorical columns. if empty metric will consider all features as numeric

# RETURN a measure of stability (T-Exp, SHAP, LIME) from multiple runs over non-perturbed x.
# the greater the value, the less stable the method is

def run_stability(model, data, labels, descriptor, cat_fts=[], is_model_NN:bool=False):
    
    tensor_train= torch.from_numpy(data.values)
    tensor_labels= torch.from_numpy(labels.values.ravel().astype(int))
    
    # ------------ data reduction for testing
    tensor_train = tensor_train[0:100, :]
    tensor_labels= tensor_labels[0:100]
    # ---------------------------------------
    
    t_stability_ratios= []
    shap_stability_ratios= []
    lime_stability_ratios= []
    
    if (is_model_NN==True):
        # convert a scikit-learn NN model to a PyTorch NN model used in captum
        nn_pytorch_model= sklearn_to_pytorch_NN(model, data.shape[1])
            
        itGd_stability_ratios= []
        iXGd_stability_ratios= []
        dLif_stability_ratios= []
        lwrp_stability_ratios= []
        
        
    ohe_model= clone(model)
    
    if (np.asarray(cat_fts).shape[0]> 0):
        num_ohe_data= ohe_cat_to_numerical_simulator(data, cat_fts, delta=descriptor['ohe_delta'], 
                                                     rand_seed=True)
        ohe_model.fit(num_ohe_data, labels.values.ravel())
        retrained= True
    else:
        ohe_model.fit(data, labels.values.ravel())
        retrained= False
        
    
    runs= int(descriptor['num_runs'])
    
    for i_data, x_data in enumerate(tensor_train):
        # here x_data is tensor_train[i_data]

        # x_data and its label as pd.DataFrame
        target_i= pd.DataFrame(x_data, data.columns).T
        target_l= pd.DataFrame(data=[np.int64(tensor_labels[i_data])], columns=labels.columns)

        t_x_exps= []
        shap_x_exps= []
        lime_x_exps= []
        
        if (is_model_NN==True):
            itGd_x_exps= []
            iXGd_x_exps= []
            dLif_x_exps= []
            lwrp_x_exps= []
                               
        for i in range(runs):
            # ignore temporarily warnings related to feature names
            with warnings.catch_warnings():
                warnings.filterwarnings("ignore", message="X does not have valid feature names")
            
                # ------------------------------------ n runs x_data explanation
                # data point explanation -- T-Exp
                t_x_exp, t_x_ft, t_x_sh= categorical_taylor_explainer(ohe_model, data, labels, 
                                                                      target_i, target_l, cat_cols=cat_fts,
                                              h_min=descriptor['h_min'], h_max=descriptor['h_max'],
                                              eps=descriptor['jacobian_eps'], max_itr=descriptor['max_itr'],
                                              finite_diff_version=descriptor['finite_diff_version'],
                                              delta=descriptor['ohe_delta'], e_x=descriptor['e_x'], 
                                              angle=descriptor['angle'], retrained_ohe_model=retrained, 
                                              verbose=False)

                # data point explanation -- SHAP
                if isinstance(model, xgb.XGBModel):
                    shap_exp_gen= shap.TreeExplainer(model)
                else:
                    shap_exp_gen= shap.Explainer(model.predict, data)
                
                shap_x_exp= shap_exp_gen(target_i)
                shap_x_exp= shap_x_exp.values


                # data point explanation -- LIME
                lime_exp_gen= lime.lime_tabular.LimeTabularExplainer(training_data=np.asarray(data), 
                                                                     feature_names=np.asarray(data.columns), 
                                                                     training_labels=labels.values.ravel().astype(int), 
                                                                     class_names=np.asarray([0,1]), 
                                                                     mode='classification', 
                                                                     discretize_continuous=False, verbose=False)

                lime_scores= lime_exp_gen.explain_instance(data_row=np.asarray(target_i)[0], 
                                                           predict_fn=model.predict_proba, 
                                                           num_features=data.shape[1])

                lime_x_exp= lime_exp_in_data_order(lime_scores, data.shape[1])
            
            # reset the warning settings
            warnings.resetwarnings()
            
            
            # data point explanation -- Gradient-based methods
            if (is_model_NN==True):
                x_data_tensor= torch.tensor(np.asarray(target_i), dtype=torch.float32)
                x_data_tensor.requires_grad= True

                # ignore non-important warnings temporarily
                with warnings.catch_warnings():
                    warnings.filterwarnings("ignore", message="Setting forward, backward hooks and attributes")

                    itGd= IntegratedGradients(nn_pytorch_model)
                    itGd_x_exp= (itGd.attribute(x_data_tensor)).squeeze().detach()

                    iXGd= InputXGradient(nn_pytorch_model)
                    iXGd_x_exp= (iXGd.attribute(x_data_tensor.unsqueeze(0))).squeeze().detach()

                    dLif= DeepLift(nn_pytorch_model)
                    dLif_x_exp= (dLif.attribute(x_data_tensor.unsqueeze(0))).squeeze().detach()

                    lwrp= LRP(nn_pytorch_model)
                    lwrp_x_exp= (lwrp.attribute(x_data_tensor.unsqueeze(0))).squeeze().detach()

                # reset the warning settings
                warnings.resetwarnings()
            
            # get the explanation of each method to each run            
            t_x_exps.append(t_x_exp)
            shap_x_exps.append(shap_x_exp.squeeze())
            lime_x_exps.append(lime_x_exp)
            
            if (is_model_NN==True):
                itGd_x_exps.append(itGd_x_exp.numpy())
                iXGd_x_exps.append(iXGd_x_exp.numpy())
                dLif_x_exps.append(dLif_x_exp.numpy())
                lwrp_x_exps.append(lwrp_x_exp.numpy())
                
                
        t_x_exps= np.asarray(t_x_exps)
        shap_x_exps= np.asarray(shap_x_exps)
        lime_x_exps= np.asarray(lime_x_exps)
            
        t_x_exps_mean= np.mean(t_x_exps, axis=0)
        shap_x_exps_mean= np.mean(shap_x_exps, axis=0)
        lime_x_exps_mean= np.mean(lime_x_exps, axis=0)
        
        t_x_exp_ratios= []
        shap_x_exp_ratios= []
        lime_x_exp_ratios= []
        
        if (is_model_NN==True):
            itGd_x_exps= np.asarray(itGd_x_exps)
            iXGd_x_exps= np.asarray(iXGd_x_exps)
            dLif_x_exps= np.asarray(dLif_x_exps)
            lwrp_x_exps= np.asarray(lwrp_x_exps)
            
            itGd_x_exps_mean= np.mean(itGd_x_exps, axis=0)
            iXGd_x_exps_mean= np.mean(iXGd_x_exps, axis=0)
            dLif_x_exps_mean= np.mean(dLif_x_exps, axis=0)
            lwrp_x_exps_mean= np.mean(lwrp_x_exps, axis=0)
            
            itGd_x_exp_ratios= []
            iXGd_x_exp_ratios= []
            dLif_x_exp_ratios= []
            lwrp_x_exp_ratios= []
            

        # ------------------------------------ distance of each explanation from the mean of explanations 
        for j in range(runs):
            t_x_exp_ratios.append(lp_norm_dif(t_x_exps_mean, t_x_exps[j], 
                                              p_norm=descriptor['p_norm'], norm=False))

            shap_x_exp_ratios.append(lp_norm_dif(shap_x_exps_mean, shap_x_exps[j], 
                                                 p_norm=descriptor['p_norm'], norm=False))

            lime_x_exp_ratios.append(lp_norm_dif(lime_x_exps_mean, lime_x_exps[j], 
                                                 p_norm=descriptor['p_norm'], norm=False))
            
            if (is_model_NN==True):
                itGd_x_exp_ratios.append(lp_norm_dif(itGd_x_exps_mean, itGd_x_exps[j], 
                                              p_norm=descriptor['p_norm'], norm=False))
                
                iXGd_x_exp_ratios.append(lp_norm_dif(iXGd_x_exps_mean, iXGd_x_exps[j], 
                                              p_norm=descriptor['p_norm'], norm=False))
                
                dLif_x_exp_ratios.append(lp_norm_dif(dLif_x_exps_mean, dLif_x_exps[j], 
                                              p_norm=descriptor['p_norm'], norm=False))
                
                lwrp_x_exp_ratios.append(lp_norm_dif(lwrp_x_exps_mean, lwrp_x_exps[j], 
                                              p_norm=descriptor['p_norm'], norm=False))
        
            
        # ------------------------------------ max ratio related to each x_data
        t_stability_ratios.append(t_x_exp_ratios[np.argmax(t_x_exp_ratios)])
        shap_stability_ratios.append(shap_x_exp_ratios[np.argmax(shap_x_exp_ratios)])
        lime_stability_ratios.append(lime_x_exp_ratios[np.argmax(lime_x_exp_ratios)])
        
        if (is_model_NN==True):
            itGd_stability_ratios.append(itGd_x_exp_ratios[np.argmax(itGd_x_exp_ratios)])
            iXGd_stability_ratios.append(iXGd_x_exp_ratios[np.argmax(iXGd_x_exp_ratios)])
            dLif_stability_ratios.append(dLif_x_exp_ratios[np.argmax(dLif_x_exp_ratios)])
            lwrp_stability_ratios.append(lwrp_x_exp_ratios[np.argmax(lwrp_x_exp_ratios)])
        
        """
        print('instance number', i_data)
        print('t_x_exps\n', t_x_exps)
        print('shap_x_exps\n', shap_x_exps)
        print('lime_x_exps\n', lime_x_exps)
        print('t_x_exps_mean\n', t_x_exps_mean)
        print('shap_x_exps_mean\n', shap_x_exps_mean)
        print('lime_x_exps_mean\n', lime_x_exps_mean)
        print('t_x_exp_ratios\n', t_x_exp_ratios)
        print('shap_x_exp_ratios\n', shap_x_exp_ratios)
        print('lime_x_exp_ratios\n', lime_x_exp_ratios)
        print('t_stab_ratios\n', t_stability_ratios)
        print('shap_stab_ratios\n', shap_stability_ratios)
        print('lime_stab_ratios\n', lime_stability_ratios)
        """
    
    # ------------------------------------ general max ratio related to all data
    t_max = t_stability_ratios[np.argmax(t_stability_ratios)]
    shap_max= shap_stability_ratios[np.argmax(shap_stability_ratios)]
    lime_max= lime_stability_ratios[np.argmax(lime_stability_ratios)]
    
    if (is_model_NN==True):
        itGd_max= itGd_stability_ratios[np.argmax(itGd_stability_ratios)]
        iXGd_max= iXGd_stability_ratios[np.argmax(iXGd_stability_ratios)]
        dLif_max= dLif_stability_ratios[np.argmax(dLif_stability_ratios)]
        lwrp_max= lwrp_stability_ratios[np.argmax(lwrp_stability_ratios)]
        
        # max stability_ratios
        return t_max, shap_max, lime_max, itGd_max, iXGd_max, dLif_max, lwrp_max

    # max stability_ratios
    return t_max, shap_max, lime_max